#### Stream Customers data from Cloud files to Delta Lake
    1. Read files from Cloud storgae using DataStreamReader API
    2. Transform the dataframe to add the following columns
            filepath: Cloud file path
            ingestiondate: Current Timestamp
    3. Write the transformed data stream to Delata Lake Table

##### 1. Read files from Cloud storgae using DataStreamReader API

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

customers_schema = StructType(fields = [StructField('customer_id', IntegerType()),
                                        StructField('customer_name', StringType()),
                                        StructField('date_of_birth', DateType()),
                                        StructField('telephone', StringType()),
                                        StructField('email', StringType()),
                                        StructField('member_since', DateType()),
                                        StructField('created_timestamp', TimestampType())                                        
                                        ]
                            )

In [0]:
customers_df = (
                    spark.readStream
                        .format("json")
                        .schema(customers_schema)
                        .load("/Volumes/gizmobox_catalog_subbu/landing/operational_data_volume/customers_stream/")
)

##### 2.Transform the dataframe to add the following columns
-  filepath: Cloud file path
-  ingestiondate: Current Timestamp



In [0]:
from pyspark.sql.functions import col, current_timestamp
customers_transformed_df = (
                            customers_df
                                .withColumn("file_path", col("_metadata.file_path"))
                                .withColumn("ingestion_date", current_timestamp())
)

##### 3. Write the transformed data stream to Delata Lake Table

In [0]:
streaming_query = ( customers_transformed_df.writeStream
                        .format("delta")
                        .outputMode("append")
                        .trigger(processingTime="60 seconds")
                        .option("checkpointLocation", "/Volumes/gizmobox_catalog_subbu/landing/operational_data_volume/customers_stream/_checkpoint_stream")
                        .toTable("gizmobox_catalog_subbu.bronze.customers_stream")
)

In [0]:
streaming_query.stop()

In [0]:
%sql
select * from gizmobox_catalog_subbu.bronze.customers_stream